# 💧 Irrigation Recommendation System — AI Model Training

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, BaggingClassifier,
    GradientBoostingClassifier, AdaBoostClassifier, StackingClassifier
)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

## Step 1: Load & Explore Data

In [ ]:
df = pd.read_csv('Datasets/Irrigation_recommendation.csv')
print('Shape:', df.shape)
print('\nClass distribution:')
print(df['irrigation_schedule'].value_counts())
df.head()

In [ ]:
print(df.isnull().sum())
df.describe()

## Step 2: EDA

In [ ]:
fig = px.histogram(df, x='irrigation_schedule', color='irrigation_schedule',
                   title='Irrigation Schedule Distribution')
fig.show()

In [ ]:
num_cols = ['soil_moisture', 'evapotranspiration', 'temperature', 'humidity', 'rainfall', 'ph']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flatten(), num_cols):
    df.boxplot(column=col, by='irrigation_schedule', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('')
plt.suptitle('Feature distributions by Irrigation Schedule')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()

## Step 3: Preprocessing

In [ ]:
le_soil = LabelEncoder().fit(df['soil_type'])
le_crop = LabelEncoder().fit(df['crop_type'])

X = df.drop('irrigation_schedule', axis=1).copy()
X['soil_type'] = le_soil.transform(X['soil_type'])
X['crop_type'] = le_crop.transform(X['crop_type'])
y = df['irrigation_schedule']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print('Train:', X_train.shape, '| Test:', X_test.shape)

## Step 4: Train Base Models

In [ ]:
base_models = {
    'LogisticRegression':        LogisticRegression(solver='liblinear', max_iter=1000, random_state=42),
    'GaussianNB':                GaussianNB(),
    'SVC':                       SVC(probability=True, random_state=42),
    'KNeighborsClassifier':      KNeighborsClassifier(n_neighbors=5),
    'DecisionTreeClassifier':    DecisionTreeClassifier(random_state=42),
    'ExtraTreeClassifier':       ExtraTreeClassifier(random_state=42),
    'RandomForestClassifier':    RandomForestClassifier(n_estimators=100, random_state=42),
    'BaggingClassifier':         BaggingClassifier(random_state=42),
    'GradientBoostingClassifier':GradientBoostingClassifier(random_state=42),
    'AdaBoostClassifier':        AdaBoostClassifier(random_state=42),
    'CatBoostClassifier':        CatBoostClassifier(verbose=0, random_state=42),
    'LGBMClassifier':            LGBMClassifier(verbose=-1, random_state=42),
}

results = []
for name, model in base_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results.append({
        'Model': name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'Recall':    recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'F1-Score':  f1_score(y_test, y_pred, average='weighted', zero_division=0),
    })

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
results_df

In [ ]:
fig = px.bar(results_df, x='Model', y='Accuracy', color='Accuracy',
             title='Model Accuracy Comparison', color_continuous_scale='blues')
fig.update_xaxes(tickangle=30)
fig.show()

## Step 5: Stacking Ensemble

In [ ]:
meta_model = LogisticRegression(solver='liblinear', max_iter=1000, random_state=42)

stacking = StackingClassifier(
    estimators=list(base_models.items()),
    final_estimator=meta_model,
    cv=5,
    passthrough=False
)
stacking.fit(X_train, y_train)

y_pred_stack = stacking.predict(X_test)
print('Stacking Accuracy:', accuracy_score(y_test, y_pred_stack))
print(classification_report(y_test, y_pred_stack))

## Step 6: Cross-Validation

In [ ]:
cv_scores = cross_val_score(stacking, X, y, cv=5, scoring='accuracy')
print(f'CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## Step 7: Feature Importance

In [ ]:
rf_model = base_models['RandomForestClassifier']
feat_imp = pd.DataFrame({'Feature': X.columns, 'Importance': rf_model.feature_importances_})
feat_imp = feat_imp.sort_values('Importance', ascending=False)

fig = px.bar(feat_imp, x='Feature', y='Importance', color='Importance',
             title='Feature Importance (Random Forest)', color_continuous_scale='greens')
fig.show()

## Step 8: Save Model

In [ ]:
# Re-train base models on full training data for saving
trained_base_models = {}
for name, model in base_models.items():
    model.fit(X_train, y_train)
    trained_base_models[name] = model

# Extract the fitted meta-model from stacking
fitted_meta = stacking.final_estimator_

model_bundle = {
    'base_models': trained_base_models,
    'meta_model':  fitted_meta,
    'le_soil':     le_soil,
    'le_crop':     le_crop,
    'feature_names': list(X.columns)
}

joblib.dump(model_bundle, 'Models/Irrigation_Recommendation_model.pkl')
print('✅ Model saved to Models/Irrigation_Recommendation_model.pkl')